# Analyse des MixUp Regressors (Style-Score)
### Masterarbeit: Automatische Übersetzung von Alltagssprache (AS) in Leichte Sprache (LS)

Dieses Notebook evaluiert den trainierten **MixUp-Regressor**, der den Grad der Leichten Sprache $\lambda \in [0, 1]$ schätzt.
1. Laden des Vokabulars aus `data/vocabs/mixup_vocab.json`.
2. Laden des Modells `bilstm_mixup_regression.pt`.
3. Evaluierung auf Test- und Validierungsdaten.
4. Analyse des Verhaltens bei benutzerdefinierten Sätzen.


In [8]:
import os
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import spacy
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns

# Arbeitsverzeichnis auf Projekt-Root setzen
while not os.path.exists(".git"):
    parent = os.path.dirname(os.getcwd())
    if parent == os.getcwd():
        break
    os.chdir("..")
print("Arbeitsverzeichnis:", os.getcwd())


Arbeitsverzeichnis: /Users/fietescheel/Documents/Master Thesis


In [9]:
VOCAB_PATH = "data/new_pipeline/vocabs/mixup_vocab.json"
MODEL_PATH = "results/models/new_pipeline/bilstm_mixup_regression.pt"
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
MAX_SEQ_LEN = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print("Nutze Device:", DEVICE)


Nutze Device: mps


## 1. Vokabular und Modell laden


In [10]:
class BiLSTMRegressor(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout=0.3):
        super(BiLSTMRegressor, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)
        out = self.fc(self.dropout(hidden))
        return self.sigmoid(out)

# Vokabular laden
with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    stoi = json.load(f)
print(f"Vokabular-Größe geladen: {len(stoi)}")

model = BiLSTMRegressor(len(stoi), EMBEDDING_DIM, HIDDEN_DIM).to(DEVICE)
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    print("Modell geladen!")
else:
    print(f"WARNUNG: Pfad {MODEL_PATH} nicht gefunden!")
model.eval()


Vokabular-Größe geladen: 25002
WARNUNG: Pfad results/models/new_pipeline/bilstm_mixup_regression.pt nicht gefunden!


BiLSTMRegressor(
  (embedding): Embedding(25002, 128, padding_idx=0)
  (lstm): LSTM(128, 128, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (sigmoid): Sigmoid()
)

## 2. Eigene Texte bewerten (Stil-Komplexitäts-Score)
Der Score liegt zwischen 0 (komplett Alltagssprache) und 1 (perfekt Leichte Sprache).


In [11]:
nlp = spacy.load("de_core_news_sm", disable=["ner", "tagger", "lemmatizer"])

def predict_style_score(text):
    tokens = [t.text.lower() for t in nlp(text) if not t.is_space]
    encoded = [stoi.get(t, stoi.get("<unk>", 1)) for t in tokens][:MAX_SEQ_LEN]
    padded = encoded + [0] * (MAX_SEQ_LEN - len(encoded))
    inp = torch.tensor([padded], dtype=torch.long).to(DEVICE)
    
    with torch.no_grad():
        score = model(inp).item()
    return score

as_bsp = "Es ist ratsam, die Formalitäten unverzüglich abzuschließen."
ls_bsp = "Machen Sie die Papiere schnell fertig."

print(f"AS Satz: '{as_bsp}' -> Score: {predict_style_score(as_bsp):.4f}")
print(f"LS Satz: '{ls_bsp}' -> Score: {predict_style_score(ls_bsp):.4f}")


AS Satz: 'Es ist ratsam, die Formalitäten unverzüglich abzuschließen.' -> Score: 0.4866
LS Satz: 'Machen Sie die Papiere schnell fertig.' -> Score: 0.4911
